# NoProp Ternary — Results Aggregator

Run this notebook **locally** after all parallel platform notebooks finish.

## Inputs required
Collect the `run_report.json` and `summary.csv` from each platform run and place them in:
```
aggregated_results/
  colab/          <- 0A + 1A
    run_report.json
    summary.csv
    0A_fp32_noprop_logs.csv
    1A_noprop_ste_logs.csv
  kaggle/         <- 0B + 1B (tab 1)
    run_report.json
    summary.csv
  kaggle2/        <- 0C + 1C (tab 2)
    run_report.json
    summary.csv
  lightning/      <- 1D
    run_report.json
    summary.csv
  kaggle_5k/      <- All 7 @ 5k steps (12 h session)
    run_report.json
    summary.csv
```

**Tip:** Each platform notebook saves `run_report.json` + `summary.csv` at the end.
Download and place in subfolders above.

## Platform roster (all free)

| Platform | File | Experiments | Cost |
|---|---|---|---|
| Colab Free (T4) | `noprop_colab.ipynb` | 0A + 1A | Free |
| Kaggle tab 1 (P100) | `noprop_kaggle.ipynb` | 0B + 1B | Free |
| Kaggle tab 2 (P100) | `noprop_kaggle2.ipynb` | 0C + 1C | Free |
| Lightning AI Free (T4) | `noprop_lightning.ipynb` | 1D | Free |
| Kaggle 12 h session | `noprop_kaggle_5k.ipynb` | All 7 @ 5k steps | Free |

## What this produces
1. Unified comparison table - all 7 experiments, val_ppl, dead_rate, pass/fail, elapsed time, platform
2. Cross-platform consistency check - same experiment on multiple platforms should agree within 10%
3. Plots - val_ppl and dead_rate learning curves, all experiments on one chart
4. Wave 0 vs Wave 1 cost breakdown - NoProp overhead and quantization overhead
5. Wave 2 go/no-go gate - automatic recommendation based on gate results


In [ ]:
import json, math, pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Point this at the folder containing platform subfolders ────────────────────
RESULTS_DIR = Path("aggregated_results")

if not RESULTS_DIR.exists():
    candidates = [
        Path("~/Downloads/drex_results").expanduser(),
        Path("/content/drive/MyDrive/drex_parallel_results"),
        Path("/kaggle/working/drex_parallel_results"),
        Path.home() / "drex_parallel_results",
    ]
    for c in candidates:
        if c.exists():
            RESULTS_DIR = c
            break

print("Scanning:", RESULTS_DIR)
reports  = sorted(RESULTS_DIR.rglob("run_report.json"))
csv_logs = sorted(RESULTS_DIR.rglob("*_logs.csv"))
print(f"Reports  : {len(reports)}")
print(f"Log CSVs : {len(csv_logs)}")
for r in reports:
    print(f"  {r.relative_to(RESULTS_DIR)}")


In [ ]:
# ── Load and merge all reports ────────────────────────────────────────────────
records = []
for rpath in reports:
    with open(rpath) as f:
        rep = json.load(f)
    platform = rep.get("platform", rpath.parent.name)
    for e in rep.get("experiments", []):
        e["platform"] = platform
        e["run_name"] = rep.get("run_name", "?")
        e["device"]   = rep.get("device",   "?")
        records.append(e)

all_df = pd.DataFrame(records)
print(f"Loaded {len(all_df)} experiment results from {all_df['platform'].nunique()} platform(s)")
print(all_df[["exp_id","platform","val_ppl","dead_rate","elapsed_sec"]].to_string(index=False))


In [ ]:
# ── Unified comparison table ──────────────────────────────────────────────────
GATE_PPL  = {"0A_fp32_noprop":250, "0B_ternary_backprop":250, "0C_fp32_backprop":200,
             "1A_noprop_ste":300,  "1B_noprop_trust":300, "1C_noprop_dqt":300, "1D_noprop_hestia":300}
GATE_DEAD = {"0A_fp32_noprop":0.30,"0B_ternary_backprop":0.15,"0C_fp32_backprop":0.10,
             "1A_noprop_ste":0.30, "1B_noprop_trust":0.30,"1C_noprop_dqt":0.30,"1D_noprop_hestia":0.30}
EXP_ORDER = ["0A_fp32_noprop","0B_ternary_backprop","0C_fp32_backprop",
             "1A_noprop_ste","1B_noprop_trust","1C_noprop_dqt","1D_noprop_hestia"]

# Keep highest-step result if an experiment appears on multiple platforms
deduped = all_df.sort_values("step", ascending=False).drop_duplicates(subset="exp_id", keep="first") if "step" in all_df.columns else all_df.drop_duplicates(subset="exp_id", keep="first")

rows = []
for eid in EXP_ORDER:
    row = deduped[deduped["exp_id"] == eid]
    if row.empty:
        rows.append({"exp_id":eid,"platform":"-","device":"-",
                     "val_ppl":float("nan"),"dead_rate":float("nan"),
                     "elapsed_min":float("nan"),"gate":"NOT RUN"}); continue
    r = row.iloc[0]
    ppl  = float(r.get("val_ppl",  float("nan")))
    dead = float(r.get("dead_rate",float("nan")))
    ok   = (math.isfinite(ppl) and ppl < GATE_PPL.get(eid,999)) and            (math.isfinite(dead) and dead < GATE_DEAD.get(eid,1.0))
    elapsed = float(r.get("elapsed_sec", float("nan")))
    rows.append({"exp_id":eid,"platform":r.get("platform","-"),"device":r.get("device","-"),
                 "val_ppl":round(ppl,2),"dead_rate":round(dead,4),
                 "elapsed_min":round(elapsed/60,1) if math.isfinite(elapsed) else float("nan"),
                 "gate_ppl":GATE_PPL.get(eid,"?"),"gate_dead":GATE_DEAD.get(eid,"?"),
                 "gate":"PASS" if ok else "FAIL"})

summary_agg = pd.DataFrame(rows)
pd.set_option("display.max_columns",20,"display.width",120)

print("=" * 90)
print("UNIFIED COMPARISON TABLE — All 7 Experiments")
print("=" * 90)
print(summary_agg[["exp_id","platform","val_ppl","dead_rate","elapsed_min","gate_ppl","gate_dead","gate"]].to_string(index=False))
passing = summary_agg[summary_agg["gate"] == "PASS"]
print()
print(f"Gate results: {len(passing)}/{len(summary_agg)} passed  |  Passing: {list(passing['exp_id'])}")


In [ ]:
# ── Cross-platform consistency check ─────────────────────────────────────────
if all_df["platform"].nunique() > 1:
    dupes = all_df.groupby("exp_id").filter(lambda g: len(g) > 1)
    if not dupes.empty:
        print("Cross-platform val_ppl comparison (same exp on multiple platforms):")
        pivot = dupes.pivot_table(index="exp_id", columns="platform", values="val_ppl", aggfunc="last")
        print(pivot.round(2).to_string())
        print()
        for eid, row in pivot.iterrows():
            vals = row.dropna()
            if len(vals) > 1:
                spread = (vals.max() - vals.min()) / vals.mean()
                flag = "  <- INCONSISTENT (>10%)" if spread > 0.10 else "  OK"
                print(f"  {eid}: spread={spread:.1%}{flag}")
    else:
        print("No duplicate experiments across platforms.")
else:
    print("Only one platform loaded; no cross-platform check.")


In [ ]:
# ── Wave 0 vs Wave 1 cost breakdown ──────────────────────────────────────────
print("WAVE 0 — Baseline costs:")
w0 = summary_agg[summary_agg["exp_id"].str.startswith("0")]
print(w0[["exp_id","val_ppl","dead_rate","gate"]].to_string(index=False))
print()
print("WAVE 1 — NoProp + Ternary combinations:")
w1 = summary_agg[summary_agg["exp_id"].str.startswith("1")]
print(w1[["exp_id","val_ppl","dead_rate","gate"]].to_string(index=False))
print()

ppl_0a = summary_agg.loc[summary_agg.exp_id=="0A_fp32_noprop","val_ppl"].values
ppl_0c = summary_agg.loc[summary_agg.exp_id=="0C_fp32_backprop","val_ppl"].values
if ppl_0a.size and ppl_0c.size and math.isfinite(float(ppl_0a[0])) and math.isfinite(float(ppl_0c[0])):
    overhead = (float(ppl_0a[0]) - float(ppl_0c[0])) / float(ppl_0c[0])
    print(f"NoProp vs Backprop overhead (0A vs 0C): {overhead:+.1%} val_ppl")
    for _, row in w1[w1["val_ppl"].notna()].iterrows():
        delta = (float(row["val_ppl"]) - float(ppl_0a[0])) / float(ppl_0a[0])
        label = "OK (<20% above NoProp baseline)" if abs(delta) < 0.20 else "EXCEEDS 20% overhead"
        print(f"  {row['exp_id']} vs 0A: {delta:+.1%}  {label}")


In [ ]:
# ── Learning curve plots ─────────────────────────────────────────────────────
if csv_logs:
    curve_df = pd.concat([pd.read_csv(c).assign(platform=c.parent.name) for c in csv_logs], ignore_index=True)
    color_map = {
        "0A_fp32_noprop":"#1f77b4","0B_ternary_backprop":"#6baed6",
        "0C_fp32_backprop":"#9ecae1","1A_noprop_ste":"#d62728",
        "1B_noprop_trust":"#ff7f0e","1C_noprop_dqt":"#2ca02c","1D_noprop_hestia":"#9467bd",
    }
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for eid, g in curve_df.groupby("exp_id"):
        color = color_map.get(eid, "gray")
        ls = "-" if eid.startswith("0") else "--"
        axes[0].plot(g["step"], g["val_ppl"],   color=color, ls=ls, lw=2, label=eid)
        axes[1].plot(g["step"], g["dead_rate"],  color=color, ls=ls, lw=2, label=eid)
        axes[2].plot(g["step"], g["grad_norm"],  color=color, ls=ls, lw=2, label=eid)
    axes[0].set_title("Val Perplexity"); axes[1].set_title("Dead Neuron Rate"); axes[2].set_title("Grad Norm")
    for ax in axes: ax.set_xlabel("step"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    w0p = mpatches.Patch(color="steelblue", label="Wave 0 baseline (solid)")
    w1p = mpatches.Patch(color="red",       label="Wave 1 NoProp+ternary (dashed)")
    fig.legend(handles=[w0p, w1p], loc="upper right", fontsize=9)
    plt.tight_layout(); plt.savefig("aggregated_plot.png", dpi=120); plt.show()
    print("Plot saved: aggregated_plot.png")
else:
    print("No *_logs.csv files found. Download the per-experiment CSVs from each platform and place them alongside run_report.json.")


In [ ]:
# ── Wave 2 go/no-go recommendation ───────────────────────────────────────────
print("=" * 65)
print("WAVE 2 GO / NO-GO GATE")
print("=" * 65)
w1_pass = summary_agg[(summary_agg.exp_id.str.startswith("1")) & (summary_agg.gate=="PASS")]

if len(w1_pass) == 0:
    print("RESULT : NO-GO")
    print("No Wave-1 variant passed all gates.")
    print()
    print("Checklist:")
    print("  1. Did all 4 free-tier notebooks complete successfully?")
    print("  2. Re-run any NOT RUN experiments from the table above.")
    print("  3. If val_ppl gate failed: try steps=2000 on the same platform.")
    print("  4. If dead_rate gate failed: lower lr to 1e-4 and re-run.")
elif len(w1_pass) == 1:
    best = w1_pass.iloc[0]["exp_id"]
    print(f"RESULT : CONDITIONAL GO  (only {best} passed)")
    print(f"Proceed to Wave 2 with {best} only.")
    print("Open noprop_ternary_scale_runs.ipynb:")
    mode = best.split("_")[-1]
    print(f"  NOPROP_MODES = ['{mode}']")
    print(f"  STEPS_NOPROP = 5000")
else:
    best = w1_pass.sort_values("val_ppl").iloc[0]
    modes = list(w1_pass.sort_values("val_ppl")["exp_id"].str.replace(r"1[A-D]_noprop_","",regex=True))
    print(f"RESULT : GO  ({len(w1_pass)}/4 variants passed)")
    print(f"Best variant: {best['exp_id']}  val_ppl={best['val_ppl']:.1f}")
    print()
    print("Open noprop_ternary_scale_runs.ipynb:")
    print(f"  NOPROP_MODES = {modes}")
    print(f"  STEPS_NOPROP = 5000")
    print()
    print("Passing variants (ranked by val_ppl):")
    for _, r in w1_pass.sort_values("val_ppl").iterrows():
        print(f"  {r['exp_id']:28}  val_ppl={r['val_ppl']:.1f}  dead={r['dead_rate']:.3f}")

summary_agg.to_csv("aggregated_summary.csv", index=False)
print()
print("Full summary saved: aggregated_summary.csv")
